# Matched Filter for Gravitational Wave Detection — ripple

This notebook implements a **matched filter** for detecting BNS gravitational wave signals using the [ripple](https://github.com/tedwards2412/ripple) library.

It operates on data produced by **Step 1 (data_generation)** of the SparseBank pipeline. Each HDF5 file contains pre-whitened strain from two detectors (H1, L1) along with the ground-truth source parameters.

## Data format

| Dataset | Shape | Description |
|---------|-------|-------------|
| `injected_data` | `(N, 2, 28160)` | Whitened H1+L1 strain, 55 s at 512 Hz |
| `mass_1`, `mass_2` | `(N,)` | Component masses [$M_\odot$] |
| `chirp_mass`, `mass_ratio` | `(N,)` | Derived mass parameters |
| `chi1`, `chi2` | `(N,)` | Aligned spins |
| `distance` | `(N,)` | Luminosity distance [Mpc] |
| `snr` | `(N,)` | Network SNR |

## Signal timing

The data generation pipeline injects the signal such that the merger occurs ~64 s from the start of the window. Since only the first 55 s are stored, the data contains the **pre-merger inspiral chirp**. The matched filter SNR peak will appear near the end of the analysis window.

## Outline

1. Load HDF5 data and inspect the file structure
2. Visualise the whitened strain
3. Generate a TaylorF2 template using true parameters (ripple)
4. Whiten the template with the nominal aLIGO PSD
5. Matched filter on the pre-whitened data (flat PSD)
6. Template bank search over a $(m_1, m_2)$ grid
7. Match and fitting factor
8. Differentiable SNR gradient via JAX

## 1. Imports

In [ ]:
%config InlineBackend.figure_format = 'retina'

import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import jax
import jax.numpy as jnp
from jax import config

config.update("jax_enable_x64", True)

from ripplegw.waveforms import TaylorF2
from ripplegw import ms_to_Mc_eta

print(f"JAX devices: {jax.devices()}")

## 2. Configuration

Set `HDF5_PATH` to a file produced by Step 1 of the SparseBank pipeline (e.g. `{data_dir}/test/sig_combined_test_0.h5`).

In [ ]:
# ── Path to a Step-1 HDF5 file ────────────────────────────────────────────────
HDF5_PATH = Path("path/to/data/test/sig_combined_test_0.h5")   # <── set this

# ── Pipeline parameters (must match data_generation config) ──────────────────
SAMPLE_RATE = 512    # Hz
F_LOW       = 20.0   # Hz — low-frequency cutoff
F_HIGH      = 256.0  # Hz — Nyquist (= SAMPLE_RATE / 2)
F_REF       = 50.0   # Hz — reference frequency for waveform
DATA_DUR    = 55.0   # s  — stored window length

# ── Analysis parameters ───────────────────────────────────────────────────────
# Zero-pad to T_PAD so the matched filter can search past the stored window.
# The merger is injected at ~64 s from data start; use 128 s for safety.
T_PAD = 128.0        # s
DF    = 1.0 / T_PAD  # Hz
N_PAD = int(T_PAD * SAMPLE_RATE)

# ── Event index inside the HDF5 batch ────────────────────────────────────────
EVENT_IDX = 0

## 3. Load HDF5 Data

In [ ]:
with h5py.File(HDF5_PATH, "r") as f:
    print("Datasets in file:")
    for key in f.keys():
        print(f"  {key:20s}  shape={f[key].shape}  dtype={f[key].dtype}")

    strain_all = f["injected_data"][:]          # (N, 2, n_time)
    params = {k: f[k][:] for k in f.keys() if k != "injected_data"}

print(f"\nBatch size    : {strain_all.shape[0]}")
print(f"Channels      : {strain_all.shape[1]}  (H1, L1)")
print(f"Time samples  : {strain_all.shape[2]}  ({strain_all.shape[2]/SAMPLE_RATE:.1f} s at {SAMPLE_RATE} Hz)")

In [ ]:
# ── Extract one event ─────────────────────────────────────────────────────────
strain_H1 = strain_all[EVENT_IDX, 0, :]   # H1 whitened strain
strain_L1 = strain_all[EVENT_IDX, 1, :]   # L1 whitened strain

m1_true   = float(params["mass_1"][EVENT_IDX])
m2_true   = float(params["mass_2"][EVENT_IDX])
chi1_true = float(params["chi1"][EVENT_IDX])
chi2_true = float(params["chi2"][EVENT_IDX])
dist_true = float(params["distance"][EVENT_IDX])
snr_true  = float(params["snr"][EVENT_IDX])
Mc_true, eta_true = ms_to_Mc_eta(jnp.array([m1_true, m2_true]))

print(f"Event index   : {EVENT_IDX}")
print(f"m1            : {m1_true:.3f} M_sun")
print(f"m2            : {m2_true:.3f} M_sun")
print(f"Chirp mass    : {float(Mc_true):.4f} M_sun")
print(f"chi1, chi2    : {chi1_true:.3f}, {chi2_true:.3f}")
print(f"Distance      : {dist_true:.1f} Mpc")
print(f"Network SNR   : {snr_true:.2f}")

## 4. Visualise the Whitened Strain

In [ ]:
t_data = np.arange(len(strain_H1)) / SAMPLE_RATE

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
for ax, strain, ifo in zip(axes, [strain_H1, strain_L1], ["H1", "L1"]):
    ax.plot(t_data, strain, lw=0.4, color='steelblue')
    ax.set_ylabel(f"{ifo} whitened strain")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time [s]")
fig.suptitle(
    f"Whitened strain  —  $m_1={m1_true:.2f}$, $m_2={m2_true:.2f}$ $M_\\odot$, "
    f"SNR$={snr_true:.1f}$",
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# Q-transform of H1 to visualise the chirp
from scipy.signal import spectrogram

f_spec, t_spec, Sxx = spectrogram(
    strain_H1, fs=SAMPLE_RATE,
    nperseg=256, noverlap=248,
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.pcolormesh(t_spec, f_spec, np.log1p(Sxx), shading='gouraud', cmap='inferno')
ax.set_ylim(F_LOW, 200)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Frequency [Hz]")
ax.set_title("H1 spectrogram (whitened)")
plt.tight_layout()
plt.show()

## 5. Nominal PSD for Template Whitening

The data was whitened during Step 1 using the PSD estimated from real LIGO background noise. We use a nominal aLIGO analytic PSD to apply the same whitening to the template.

After whitening, the data has approximately unit PSD, so we run the matched filter with a **flat PSD**.

In [ ]:
def aLIGO_psd(f):
    """Analytic aLIGO zero-detuned high-power PSD (Ajith & Bose 2009)."""
    f0 = 215.0
    S0 = 1e-49
    x  = f / f0
    psd = S0 * (x**(-4.14) - 5.0*x**(-2)
                + 111.0*(1.0 - x**2 + 0.5*x**4) / (1.0 + 0.5*x**2))
    psd = np.where(f < 10.0, np.inf, psd)
    psd = np.where(psd <= 0,  np.inf, psd)
    return psd


# Evaluate on the rfft grid for T_PAD
f_rfft    = np.fft.rfftfreq(N_PAD, d=1.0 / SAMPLE_RATE)
psd_rfft  = aLIGO_psd(f_rfft)
# Mask to the analysis band [F_LOW, F_HIGH]
band_mask = (f_rfft >= F_LOW) & (f_rfft <= F_HIGH)
safe_psd  = np.where(band_mask & np.isfinite(psd_rfft) & (psd_rfft > 0),
                     psd_rfft, np.inf)

f_plot = np.geomspace(10, 256, 2000)
fig, ax = plt.subplots(figsize=(8, 3))
ax.loglog(f_plot, np.sqrt(aLIGO_psd(f_plot)), color='steelblue')
ax.axvspan(F_LOW, F_HIGH, alpha=0.1, color='green', label='Analysis band')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel(r'$\sqrt{S_n(f)}$ [Hz$^{-1/2}$]')
ax.set_title('Nominal aLIGO PSD (used for template whitening)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Generate and Whiten Template (ripple)

We generate a TaylorF2 waveform using the true parameters loaded from the HDF5 file, then whiten it with the nominal aLIGO PSD:

$$\tilde{h}_w(f) = \frac{\tilde{h}(f)}{\sqrt{S_n(f)}}$$

This makes the template consistent with the pre-whitened data.

In [ ]:
# Frequency grid on which ripple evaluates the waveform
f_grid_jax = jnp.array(f_rfft[(f_rfft >= F_LOW) & (f_rfft <= F_HIGH)])

# TaylorF2 parameter vector: [Mc, eta, chi1, chi2, dist_Mpc, tc, phic, lam1, lam2]
# Use distance=1 Mpc; SNR normalisation is handled by sigma_h below
theta_true = jnp.array([
    Mc_true, eta_true,
    chi1_true, chi2_true,
    1.0,   # distance (arbitrary — normalised later)
    0.0,   # tc
    0.0,   # phic
    0.0,   # lambda1 (spins are ~0 in data_generation)
    0.0,   # lambda2
])

hp_raw, _ = TaylorF2.gen_TaylorF2_hphc(f_grid_jax, theta_true, F_REF)
hp_raw = np.array(hp_raw)
print(f"Template bins  : {len(hp_raw)}  ({len(hp_raw)*DF:.1f} s equivalent)")

# Place waveform on the full rfft grid
h_rfft = np.zeros(len(f_rfft), dtype=complex)
idx_low  = int(np.round(F_LOW  / DF))
n_template = len(hp_raw)
h_rfft[idx_low : idx_low + n_template] = hp_raw

# Whiten template
h_whitened = h_rfft / np.sqrt(safe_psd)
h_whitened = np.where(band_mask, h_whitened, 0.0 + 0.0j)

# Check whitened amplitude
print(f"Max |h_w(f)|   : {np.abs(h_whitened[band_mask]).max():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].semilogy(f_rfft[band_mask], np.abs(h_rfft[band_mask]),
                 color='steelblue', label='Raw template $|\\tilde{h}(f)|$')
axes[0].set_ylabel('Amplitude [Hz$^{-1}$]')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)
axes[0].set_title(
    f'TaylorF2 template — $m_1={m1_true:.2f}$, $m_2={m2_true:.2f}$ $M_\\odot$'
)

axes[1].semilogy(f_rfft[band_mask], np.abs(h_whitened[band_mask]),
                 color='darkorange', label='Whitened template $|\\tilde{h}_w(f)|$')
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Whitened amplitude')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Matched Filter

Because the data is pre-whitened, the matched filter uses a **flat PSD**:

$$z(t) = 4 \int_{f_{\rm low}}^{f_{\rm high}} \tilde{h}_w^*(f)\, \tilde{s}_w(f)\, e^{2\pi i f t}\, df
\qquad\Longrightarrow\qquad
\rho(t) = \frac{|z(t)|}{\sigma_h}$$

$$\sigma_h^2 = 4 \int_{f_{\rm low}}^{f_{\rm high}} |\tilde{h}_w(f)|^2\, df$$

The data is zero-padded from 55 s to 128 s so the SNR time series extends beyond the stored window to reach the expected coalescence time (~64 s).

In [ ]:
def matched_filter_whitened(strain, h_w, df, N):
    """
    Matched filter for pre-whitened data with a whitened template.

    Parameters
    ----------
    strain : 1-D real array, length N
        Pre-whitened time-domain strain (zero-padded to length N).
    h_w : complex array, length N//2+1
        Whitened frequency-domain template on the rfft grid.
    df : float
        Frequency resolution [Hz].
    N : int
        FFT length.

    Returns
    -------
    snr_t   : real array, shape (N,)  — SNR time series |z(t)| / sigma_h
    sigma_h : float                   — template normalisation
    """
    # FFT of the whitened data
    s_f = np.fft.rfft(strain) / SAMPLE_RATE

    # Template norm: sigma_h^2 = 4 * sum(|h_w|^2) * df
    sigma_h = np.sqrt(4.0 * np.sum(np.abs(h_w)**2).real * df)

    # Complex SNR in time domain via IFFT
    z_f = 4.0 * np.conj(h_w) * s_f
    z_t = np.fft.irfft(z_f, n=N) * (N * df)

    return np.abs(z_t) / sigma_h, sigma_h


# Zero-pad data to N_PAD
n_data     = len(strain_H1)
strain_pad = np.zeros(N_PAD)
strain_pad[:n_data] = strain_H1

snr_t, sigma_h = matched_filter_whitened(strain_pad, h_whitened, DF, N_PAD)
t_axis = np.arange(N_PAD) / SAMPLE_RATE

# Peak within the full time series
peak_idx  = np.argmax(snr_t)
peak_snr  = snr_t[peak_idx]
peak_time = t_axis[peak_idx]

print(f"Peak SNR       : {peak_snr:.2f}")
print(f"Peak time      : {peak_time:.3f} s")
print(f"(merger expected ~64 s from data start; data stored 0–55 s)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Top: full SNR time series
axes[0].plot(t_axis, snr_t, lw=0.6, color='steelblue')
axes[0].axvline(DATA_DUR, color='grey', ls='--', lw=1.2, label=f'End of stored data ({DATA_DUR} s)')
axes[0].axvline(peak_time, color='tomato', ls=':', lw=1.5, label=f'SNR peak at {peak_time:.2f} s')
axes[0].set_ylabel('SNR')
axes[0].set_title('Matched Filter SNR — ripple (full time series, zero-padded)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bottom: zoom around the peak
zoom_win = 5.0
t_lo = max(0, peak_time - zoom_win)
t_hi = min(T_PAD, peak_time + zoom_win)
mask_zoom = (t_axis >= t_lo) & (t_axis <= t_hi)
axes[1].plot(t_axis[mask_zoom], snr_t[mask_zoom], lw=1.0, color='steelblue')
axes[1].axvline(DATA_DUR, color='grey', ls='--', lw=1.2)
axes[1].axvline(peak_time, color='tomato', ls=':', lw=1.5)
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('SNR')
axes[1].set_title(f'Zoom around SNR peak (±{zoom_win} s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Template Bank Search

We run the matched filter over a grid of $(m_1, m_2)$ templates and record the peak SNR for each.

In [ ]:
m1_grid = np.linspace(1.0, 2.5, 10)
m2_grid = np.linspace(1.0, 2.3, 9)

bank_results = []   # (m1, m2, peak_snr, peak_time)

for m1_t in m1_grid:
    for m2_t in m2_grid:
        if m2_t > m1_t:
            continue

        Mc_t, eta_t = ms_to_Mc_eta(jnp.array([m1_t, m2_t]))
        theta_t = jnp.array([Mc_t, eta_t, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0])

        hp_t, _ = TaylorF2.gen_TaylorF2_hphc(f_grid_jax, theta_t, F_REF)
        hp_t = np.array(hp_t)

        h_t_rfft = np.zeros(len(f_rfft), dtype=complex)
        h_t_rfft[idx_low : idx_low + len(hp_t)] = hp_t[:len(h_rfft[idx_low:])]
        h_t_w = h_t_rfft / np.sqrt(safe_psd)
        h_t_w = np.where(band_mask, h_t_w, 0.0 + 0.0j)

        snr_t_bank, _ = matched_filter_whitened(strain_pad, h_t_w, DF, N_PAD)

        local_peak = snr_t_bank.max()
        local_time = t_axis[np.argmax(snr_t_bank)]
        bank_results.append((m1_t, m2_t, float(local_peak), float(local_time)))

bank_results = np.array(bank_results)
best_idx = np.argmax(bank_results[:, 2])
best = bank_results[best_idx]

print(f"Best template  : m1 = {best[0]:.2f}, m2 = {best[1]:.2f} M_sun")
print(f"Best SNR       : {best[2]:.2f}")
print(f"True params    : m1 = {m1_true:.2f}, m2 = {m2_true:.2f} M_sun  (SNR={snr_true:.1f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(bank_results[:, 0], bank_results[:, 1],
                c=bank_results[:, 2], cmap='viridis', s=80,
                edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=ax, label='Peak SNR')
ax.scatter(m1_true, m2_true, marker='*', s=300, color='tomato',
           zorder=5, label='True parameters')
ax.scatter(best[0], best[1], marker='D', s=120, color='lime',
           zorder=4, label='Best template')
ax.set_xlabel(r'$m_1$ [$M_\odot$]')
ax.set_ylabel(r'$m_2$ [$M_\odot$]')
ax.set_title('Template Bank SNR Map — ripple')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Match and Fitting Factor

The **match** between two whitened waveforms (maximised over time and phase) measures how much SNR is lost when using a template with slightly different parameters:

$$\mathcal{M}(h_1, h_2) = \max_{t_c, \phi_c} \frac{\langle h_{w,1} | h_{w,2} \rangle}{\sqrt{\langle h_{w,1} | h_{w,1} \rangle \langle h_{w,2} | h_{w,2} \rangle}}$$

In [ ]:
def compute_match(h1_w, h2_w, df, N):
    """Match maximised over time and phase for two whitened templates."""
    norm1 = np.sqrt(4.0 * np.sum(np.abs(h1_w)**2).real * df)
    norm2 = np.sqrt(4.0 * np.sum(np.abs(h2_w)**2).real * df)
    z_t   = np.fft.irfft(4.0 * np.conj(h1_w) * h2_w, n=N) * (N * df)
    return np.abs(z_t).max() / (norm1 * norm2)


matches = []
for row in bank_results:
    m1_t, m2_t = row[0], row[1]
    Mc_t, eta_t = ms_to_Mc_eta(jnp.array([m1_t, m2_t]))
    theta_t = jnp.array([Mc_t, eta_t, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0])
    hp_t, _ = TaylorF2.gen_TaylorF2_hphc(f_grid_jax, theta_t, F_REF)
    hp_t = np.array(hp_t)

    h_t_rfft = np.zeros(len(f_rfft), dtype=complex)
    h_t_rfft[idx_low : idx_low + len(hp_t)] = hp_t[:len(h_rfft[idx_low:])]
    h_t_w = np.where(band_mask, h_t_rfft / np.sqrt(safe_psd), 0.0 + 0.0j)

    matches.append(compute_match(h_whitened, h_t_w, DF, N_PAD))

matches    = np.array(matches)
best_m_idx = matches.argmax()
print(f"Fitting factor : {matches.max():.4f}")
print(f"Best-match     : m1 = {bank_results[best_m_idx, 0]:.2f}, "
      f"m2 = {bank_results[best_m_idx, 1]:.2f} M_sun")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(bank_results[:, 0], bank_results[:, 1],
                c=matches, cmap='plasma', vmin=0.8, vmax=1.0,
                s=80, edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=ax, label='Match')
ax.scatter(m1_true, m2_true, marker='*', s=300, color='cyan',
           zorder=5, label='True parameters')
ax.set_xlabel(r'$m_1$ [$M_\odot$]')
ax.set_ylabel(r'$m_2$ [$M_\odot$]')
ax.set_title('Template Match Map — ripple')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Differentiable SNR Gradient via JAX

Because `ripplegw` is built on JAX, we can differentiate the SNR with respect to source parameters. This is the key capability exploited by SparseBank for gradient-based template placement.

In [ ]:
psd_jax    = jnp.array(safe_psd)
data_f_jax = jnp.array(np.fft.rfft(strain_pad) / SAMPLE_RATE)
df_jax     = jnp.float64(DF)
band_jax   = jnp.array(band_mask)

@jax.jit
def snr_scalar(theta, data_f_in, psd_in, f_grid_in, f_ref_in,
               idx_low_in, n_rfft, df_in, band_in):
    """Differentiable scalar |SNR| at fixed tc using the whitened inner product."""
    hp, _ = TaylorF2.gen_TaylorF2_hphc(f_grid_in, theta, f_ref_in)

    h_pad = jnp.zeros(n_rfft, dtype=jnp.complex128)
    h_pad = h_pad.at[idx_low_in : idx_low_in + len(hp)].set(hp)

    # Whiten
    safe = jnp.where(band_in & jnp.isfinite(psd_in) & (psd_in > 0), psd_in, jnp.inf)
    h_w  = jnp.where(band_in, h_pad / jnp.sqrt(safe), 0.0 + 0.0j)

    # Inner product <h_w | s_w> (flat PSD, evaluate at tc=0)
    integrand = jnp.conj(h_w) * data_f_in
    hs        = 4.0 * jnp.abs(jnp.sum(integrand)) * df_in
    sigma_h   = jnp.sqrt(4.0 * jnp.sum(jnp.abs(h_w)**2).real * df_in)

    return hs / sigma_h


snr_val = snr_scalar(theta_true, data_f_jax, psd_jax, f_grid_jax,
                     F_REF, idx_low, len(f_rfft), df_jax, band_jax)
print(f"Differentiable |SNR| at true params : {float(jnp.abs(snr_val)):.3f}")

grad_fn = jax.grad(lambda th: jnp.abs(
    snr_scalar(th, data_f_jax, psd_jax, f_grid_jax,
               F_REF, idx_low, len(f_rfft), df_jax, band_jax)
))

g = grad_fn(theta_true)
param_names = ['Mc', 'eta', 'chi1', 'chi2', 'dist', 'tc', 'phic', 'lam1', 'lam2']
print("\nGradient of |SNR| w.r.t. source parameters:")
for name, gi in zip(param_names, g):
    print(f"  d|SNR|/d({name:6s}) = {float(gi):+.4e}")

## Summary

### Data pipeline

| Step | Description |
|------|-------------|
| HDF5 source | `injected_data` — whitened H1+L1 strain, 55 s @ 512 Hz |
| Template | TaylorF2 via `ripplegw`, whitened with aLIGO PSD |
| Matched filter | Flat-PSD inner product (data already whitened) |
| SNR search | Zero-pad to 128 s so the peak near $t_c \approx 64$ s is visible |

### Key takeaways

* The pre-whitened HDF5 data requires a **whitened template** and a **flat PSD** for matched filtering.
* Zero-padding extends the SNR time series beyond the stored 55-second window to reveal the coalescence peak.
* **JAX differentiability** through `ripplegw` enables gradient-based template bank construction — the core contribution of SparseBank.